In [1]:
import torch
from datasets import Dataset

def parse_iob_file(file_path):
    sentences = []
    tokens = []
    tags = []
    
    # Dictionary for converting text labels to IDs
    label_map = {"O": 0, "B-MOUNTAIN": 1, "I-MOUNTAIN": 2}

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": tags})
                    tokens = []
                    tags = []
                continue
            
            # Assume that the separator is a space or tab
            splits = line.split()
            if len(splits) == 2:
                token, tag = splits[0], splits[1]
                tokens.append(token)
                tags.append(label_map.get(tag, 0)) # If the label is unknown, set it to 'O' (0)
                
        # Add the last sentence if the file did not end with a blank line
        if tokens:
            sentences.append({"tokens": tokens, "ner_tags": tags})
            
    return Dataset.from_list(sentences)

#Upload data (specify your path to the file)
raw_dataset = parse_iob_file("train.txt")
print(f"Sentences loaded: {len(raw_dataset)}")
print("Example of this:", raw_dataset[0])

Sentences loaded: 4000
Example of this: {'tokens': ['Snow', 'covered', 'Ben', 'Nevis', 'overnight', '.'], 'ner_tags': [0, 0, 1, 2, 0, 0]}


In [2]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased" # Or use multilingual bert-base-multilingual-cased if the text is in Ukrainian
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  #
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Mark special tokens like [CLS] or [SEP] as -100 (PyTorch ignores them)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                #This is the first subtoken of the new word — we give it the original label
                label_ids.append(label[word_idx])
            else:
                # These are the next subtokens of the same word.
    # For simplicity, we put the same label, or I-MOUNTAIN if it was a B-label.
    # But usually duplicating the same label works fine.
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
            
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map the function to the entire dataset
tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [3]:
import evaluate
import numpy as np

# 1. Split the dataset (e.g. 85% for training, 15% for validation)
split_dataset = tokenized_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Sample size for training: {len(train_dataset)}")
print(f"Sample size for testing: {len(val_dataset)}")

#2. Load metrics for NER
metric = evaluate.load("seqeval")

# List of tags to decode ID back to text
label_list = ["O", "B-MOUNTAIN", "I-MOUNTAIN"]

# 3. Function for calculating metrics during training
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    #Remove ignored tokens (with index -100)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Sample size for training: 3400
Sample size for testing: 600
